# Finance Graph Pipeline Tutorial

This notebook-style walkthrough mirrors the framework implementation and shows the three stages:

1. Indexing with a FIBO profile agent
2. Graph quality analysis
3. Quality-aware question answering


In [ ]:
from pathlib import Path
import json

from tutorials.finance_graph_pipeline.framework.pipeline import FinanceTutorialPipeline

base_dir = Path.cwd() / "tutorials" / "finance_graph_pipeline"
dataset_path = base_dir / "data" / "sample_finance_dataset.json"

pipeline = FinanceTutorialPipeline()
documents, questions = pipeline.load_dataset(dataset_path)
len(documents), len(questions)

## A. Indexing Stage

We first select a constrained finance profile, then run ontology-conditioned extraction, and finally materialize the graph.

In [ ]:
pipeline.index_documents(documents)

profile_summary = {
    doc_id: {
        "selected_profile": decision.selected_profile,
        "selection_confidence": decision.selection_confidence,
        "mapping_policy": decision.mapping_policy,
    }
    for doc_id, decision in pipeline.state.profile_decisions.items()
}
profile_summary

In [ ]:
{
    "nodes": {
        node_id: {
            "name": node.name,
            "entity_type": node.entity_type,
            "profile": node.profile,
            "aliases": sorted(node.aliases),
        }
        for node_id, node in pipeline.state.nodes.items()
    },
    "edges": {
        edge_id: {
            "source": edge.source_node_id,
            "relation": edge.relation_type,
            "target": edge.target_node_id,
            "profile": edge.profile,
        }
        for edge_id, edge in pipeline.state.edges.items()
    },
}

## B. Graph Quality Analysis

Next we compute both global graph quality and query-support quality.

In [ ]:
pipeline.analyze_quality(questions)

{
    "global_quality_issues": [
        {
            "issue_type": issue.issue_type,
            "severity": issue.severity,
            "message": issue.message,
            "object_id": issue.object_id,
        }
        for issue in pipeline.state.quality_issues
    ],
    "query_support": {
        question_id: {
            "answerable": record.answerable,
            "support_score": record.support_score,
            "missing_requirements": record.missing_requirements,
        }
        for question_id, record in pipeline.state.query_support.items()
    },
}

## C. Question Stage

The question stage routes each question to a profile, reads graph quality metadata, selects evidence, and generates an answer.

In [ ]:
answers = pipeline.answer_questions(questions)

[
    {
        "question_id": answer.question_id,
        "answer": answer.answer,
        "confidence": answer.confidence,
        "selected_edge_ids": answer.selected_edge_ids,
        "quality_notes": answer.quality_notes,
    }
    for answer in answers
]

## Export A Combined View

This final cell mirrors what a tutorial artifact or experiment log could look like.

In [ ]:
artifact = {
    "profile_decisions": profile_summary,
    "global_quality_issues": [issue.__dict__ for issue in pipeline.state.quality_issues],
    "query_support": {
        question_id: record.__dict__
        for question_id, record in pipeline.state.query_support.items()
    },
    "answers": [answer.__dict__ for answer in answers],
}

print(json.dumps(artifact, indent=2, default=str))